# 太陽フィラメント V52 — EdgeAttNet refiner採用版

V50(`submission_v50_no_ensemble_fold2_only.csv`, Public 0.37、V35とのA/Bで
V50≥V35を確認済み)との唯一の違いは、refinerを標準`MediSegUNet`から
`MediSegEdgeAttUNet`(EdgeAttNet論文のEdge-Guided Multi-Head Self-Attention
をボトルネックへ移植)へ差し替えた点。YOLOモデル(fold2単体、アンサンブル
なし)・後処理(shrink@0.45, drop_contained_then_area_asc)はV50と完全に同一。

新refinerは3-fold OOF独立確認済み(全結合n=707でΔ+0.0028、95%CI
[0.000915, 0.004756]、ゼロを完全に除外)。詳細は
`v51_edgeattnet_refiner_3fold_manifest.json`。

GPU必須(YOLO推論・refiner推論)。T4で可。
`MediSegEdgeAttUNet`は`solar_filament/models/mediseg_models.py`をノートへ
埋め込んであるので、Colabへ別ファイルをアップロードする必要はない。
上から順に実行する。


In [ ]:
# [1/9] 依存関係・Drive接続と前提artifactの確認
%pip install -q "ultralytics>=8.3,<9" pycocotools

import json, math, shutil
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from google.colab import drive
from pycocotools import mask as mask_utils
from tqdm.auto import tqdm
from ultralytics import YOLO

drive.mount("/content/drive")

class CFG:
    V30_ROOT = Path("/content/drive/MyDrive/solar_filament_v30")
    DATA_CACHE = V30_ROOT / "competition_data"
    ARTIFACT_ROOT = V30_ROOT / "artifacts"
    RUNS_ROOT = V30_ROOT / "runs"
    PREPROC_CACHE = V30_ROOT / "preprocessed_v30_q96"
    LOCAL_ROOT = Path("/content/solar_v52_edgeattnet")
    PREPROC_DIR = LOCAL_ROOT / "preprocessed"
    CROP_SIZE = 320
    YOLO_IMGSZ = 1280
    PREDICT_IOU = 0.55
    SINGLE_FOLD = 2  # V50と同一(OOF PQ最良、アンサンブルにとって最も厳しい比較対象)

CFG.LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
CFG.PREPROC_DIR.mkdir(parents=True, exist_ok=True)

required = {
    f"YOLO fold{CFG.SINGLE_FOLD}": CFG.RUNS_ROOT / f"yolo_fold_{CFG.SINGLE_FOLD}/weights/best.pt",
    "EdgeAttNet all-data refiner": CFG.ARTIFACT_ROOT / "mediseg_edgeattnet_refiner_all_data_resume.pt",
    "competition data": CFG.DATA_CACHE,
}
missing = {k: str(v) for k, v in required.items() if not v.exists()}
if missing:
    raise FileNotFoundError(f"前提artifactが見つかりません: {missing}")
assert torch.cuda.is_available(), "GPUランタイムを選択してください"
print({k: "OK" for k in required}, {"GPU": torch.cuda.get_device_name(0)})


In [ ]:
# [2/9] 提出パラメータ (V50と同一。差分はrefinerアーキテクチャのみ)
#
# 出典: output/v35/pipeline_stack.json の "best_config"

SUBMISSION_CONF = 0.26
SUBMISSION_OWNERSHIP_RULE = "drop_contained_then_area_asc"
REFINER_METHOD = "shrink"
REFINER_THRESHOLD = 0.45
MIN_AREA = 20

print({
    "conf": SUBMISSION_CONF, "ownership": SUBMISSION_OWNERSHIP_RULE,
    "refiner": f"{REFINER_METHOD}@{REFINER_THRESHOLD} (MediSegEdgeAttUNet)", "min_area": MIN_AREA,
    "single_fold": CFG.SINGLE_FOLD,
})
print("★ V50との唯一の差分はrefinerアーキテクチャ(EdgeAttNet採用)であることを確認 ★")


In [ ]:
# [3/9] MediSegEdgeAttUNet定義とYOLO読み込み(fold2単体のみ、アンサンブルなし)
# solar_filament/models/mediseg_models.py を V51 3-fold検証ノートと同じく埋め込む。
# Colab へ別ファイルをアップロードする必要はない。

"""
MediSeg-AI Knowledge Port for Solar Filament Segmentation
Includes:
1. MediSeg UNet
2. MediSeg DeepLabV3Plus with ASPP (Atrous Spatial Pyramid Pooling)
3. CombinedLoss (BCE + Dice Loss)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional

# ============================================================
# 1. Combined Loss (BCE + Dice Loss)
# ============================================================
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        
        intersection = (probs * targets).sum()
        dice = (2.0 * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1.0 - dice

class CombinedLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

# ============================================================
# 2. MediSeg UNet Architecture
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, mid_channels: Optional[int] = None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class MediSegUNet(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1, bilinear: bool = False):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

# ============================================================
# 3. MediSeg DeepLabV3Plus with ASPP Module
# ============================================================
class ASPP(nn.Module):
    def __init__(self, in_channels: int, out_channels: int = 256):
        super().__init__()
        self.conv1x1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1x1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=6, dilation=6, bias=False)
        self.bn3x3_1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_2 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=12, dilation=12, bias=False)
        self.bn3x3_2 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_3 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=18, dilation=18, bias=False)
        self.bn3x3_3 = nn.BatchNorm2d(out_channels)
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv_global = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn_global = nn.BatchNorm2d(out_channels)
        
        self.conv_final = nn.Conv2d(out_channels * 5, out_channels, kernel_size=1, bias=False)
        self.bn_final = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        size = x.shape[2:]
        x1 = self.relu(self.bn1x1(self.conv1x1(x)))
        x2 = self.relu(self.bn3x3_1(self.conv3x3_1(x)))
        x3 = self.relu(self.bn3x3_2(self.conv3x3_2(x)))
        x4 = self.relu(self.bn3x3_3(self.conv3x3_3(x)))
        
        x5 = self.global_avg_pool(x)
        x5 = self.relu(self.bn_global(self.conv_global(x5)))
        x5 = F.interpolate(x5, size=size, mode='bilinear', align_corners=True)
        
        x_cat = torch.cat([x1, x2, x3, x4, x5], dim=1)
        return self.relu(self.bn_final(self.conv_final(x_cat)))

class MediSegDeepLabV3Plus(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1):
        super().__init__()
        # Feature Extractor
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 512, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )
        self.aspp = ASPP(512, 256)
        self.classifier = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, out_channels, kernel_size=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_size = x.shape[2:]
        feat = self.stem(x)
        aspp_feat = self.aspp(feat)
        out = self.classifier(aspp_feat)
        return F.interpolate(out, size=input_size, mode='bilinear', align_corners=True)

# ============================================================
# 4. Edge-Guided Attention U-Net (EG-MHSA)
# ============================================================
# 2026-08-29、WINNING_STRATEGY_2026-08-29.md §1.4のPhase 2実験用に追加。
# 参考文献: "EdgeAttNet: Towards Barb-Aware Filament Segmentation"
# (arXiv:2509.02964)。MAGFiLO上でU-Net(mIoU_pairwise 0.5724)から
# EdgeAttNet(0.6451)への改善を報告しており、barb(フィラメントの微細な
# 枝構造)の認識精度向上を狙ったアーキテクチャ。本プロジェクトの
# refiner(MediSegUNet)が抱える主要な弱点(near-miss境界品質)と直接
# 対応するため、置き換え候補として追試する。
#
# 【重要な注意】著者の公開リポジトリ(github.com/dasjar/EdgeAttNet)を
# 直接確認したところ、`model/edgeattnet_model.py`(本来UNetEdgeTransformer
# クラスが定義されているべきファイル)の中身が誤って`main.py`と同一に
# なっており、実際のアーキテクチャコードはリポジトリ上に存在しない
# (2026-08-29、`gh api`でリポジトリ全ファイルを走査して確認)。学習済み
# 重みの配布も確認できなかった。**以下は論文本文のアーキテクチャ記述
# (edge prior E=σ(Conv3x3(入力画像))をbottleneckへbilinear補間し
# 1x1convで射影、Q・Kへ加算、位置エンコーディングなし)のみを根拠にした
# 独自の再実装であり、著者オリジナル実装とのコード上の一致は未検証。**
# ベースにするU-Net本体はMediSegUNetをそのまま流用しているため、
# パラメータ総数は論文記載の値(22.6M)とは一致しない。


class EdgeGuidedMHSA(nn.Module):
    """Edge-Guided Multi-Head Self-Attention (EG-MHSA)。

    U-NetのbottleneckでSelf-Attentionを行う際、入力画像から抽出した
    エッジ事前分布をQuery/Keyに加算し、境界・barb構造への感度を高める。
    Value・出力チャネル数はbottleneckの特徴マップと同一に保つ
    (MediSegUNetへdrop-inできるよう設計)。
    """

    def __init__(self, in_channels: int, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        # E = sigmoid(Conv3x3(入力画像)): 単一チャンネルのエッジ事前分布
        self.edge_conv = nn.Conv2d(in_channels, 1, kernel_size=3, padding=1)
        # E' = Conv1x1(interpolate(E)): bottleneckのチャンネル数へ射影
        self.edge_proj = nn.Conv2d(1, bottleneck_channels, kernel_size=1)
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor, original_image: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        edge_map = torch.sigmoid(self.edge_conv(original_image))
        edge_map = F.interpolate(edge_map, size=(h, w), mode="bilinear", align_corners=True)
        edge_prior = self.edge_proj(edge_map)  # (B, C, H, W)

        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        edge_flat = edge_prior.flatten(2).transpose(1, 2)  # (B, N, C)
        qk = x_flat + edge_flat  # Q = K = x + E'。位置エンコーディングは使わない(論文通り)

        attn_out, _ = self.mhsa(qk, qk, x_flat)  # V = x(エッジ由来の加算なし)
        attn_out = self.norm(attn_out + x_flat)  # 残差接続 + LayerNorm(論文に明記はないが標準的なTransformerブロックとして採用)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegEdgeAttUNet(nn.Module):
    """MediSegUNetのbottleneckにEdgeGuidedMHSAを挿入したdrop-in代替。

    入出力インターフェースはMediSegUNetと完全に同一(in_channels=3の
    crop、out_channels=1の確率マップ)。既存のrefiner学習・推論コードを
    変更せず、モデルクラスの差し替えだけで比較できるように設計している。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.edge_attn = EdgeGuidedMHSA(in_channels, bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.edge_attn(x5, x)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)


assert "MediSegEdgeAttUNet" in globals(), "MediSegEdgeAttUNet が定義されていません"

yolo_model = YOLO(str(CFG.RUNS_ROOT / f"yolo_fold_{CFG.SINGLE_FOLD}/weights/best.pt"))
refiner = MediSegEdgeAttUNet(3, 1, False).cuda().eval()
refiner_path = CFG.ARTIFACT_ROOT / "mediseg_edgeattnet_refiner_all_data_resume.pt"
try:
    refiner_checkpoint = torch.load(refiner_path, map_location="cuda", weights_only=False)
except TypeError:
    refiner_checkpoint = torch.load(refiner_path, map_location="cuda")
refiner.load_state_dict(refiner_checkpoint.get("best_model") or refiner_checkpoint["model"],
                        strict=True)
print("warm_start_from (参考):", refiner_checkpoint.get("warm_start_from"))
print({"single_fold_model": CFG.SINGLE_FOLD, "refiner_params": sum(p.numel() for p in refiner.parameters())})


In [ ]:
# [4/9] test画像一覧と単一モデル推論(アンサンブルなし。V30本体のensemble_proposalsは使わない)
def find_dataset_root(base):
    matches = list(Path(base).rglob("MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
    if not matches:
        raise FileNotFoundError(f"annotation JSONが見つからない: {base}")
    return matches[0].parents[2]

DATA_ROOT = find_dataset_root(CFG.DATA_CACHE)
TEST_DIR = next(DATA_ROOT.rglob("test_images"))
test_files = sorted(p.name for p in TEST_DIR.glob("*.jpeg"))
if len(test_files) != 180:
    raise RuntimeError(f"test画像が180枚ではありません: {len(test_files)}")
print({"test画像": len(test_files), "test_dir": str(TEST_DIR)})

for file_name in tqdm(test_files, desc="前処理画像をローカルへ"):
    cached = CFG.PREPROC_CACHE / file_name
    local = CFG.PREPROC_DIR / file_name
    if not local.exists():
        if cached.exists():
            shutil.copy2(cached, local)
        else:
            raise FileNotFoundError(f"前処理キャッシュがない: {cached}")

def result_masks(result):
    if result.masks is None:
        return []
    height, width = result.orig_shape
    output = []
    for mask in result.masks.data.detach().cpu().numpy():
        if mask.shape != (height, width):
            mask = cv2.resize(mask, (width, height), interpolation=cv2.INTER_LINEAR)
        output.append(mask >= 0.5)
    return output

def single_fold_proposals(image_path):
    """アンサンブルなし。fold2単体の生出力をそのまま返す(V35のensemble_proposalsの代わり)。"""
    result = yolo_model.predict(
        str(image_path), imgsz=CFG.YOLO_IMGSZ,
        conf=max(0.08, SUBMISSION_CONF * 0.75),
        iou=CFG.PREDICT_IOU, agnostic_nms=True, retina_masks=True, verbose=False,
    )[0]
    masks = result_masks(result)
    confidences = result.boxes.conf.detach().cpu().numpy().tolist() if result.boxes is not None else []
    keep = [(score, mask) for score, mask in zip(confidences, masks) if score >= SUBMISSION_CONF]
    return keep


In [ ]:
# [5/9] refiner適用 (V50・OOF実験と同一定義。shrink固定。変更なし)
def mask_square_box(mask, context=1.65, minimum=96):
    height, width = mask.shape
    ys, xs = np.where(mask)
    if not len(xs):
        return 0, 0, width, height
    cx, cy = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2
    side = int(math.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context))
    side = min(max(side, minimum), min(height, width))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > width: x0 -= x1 - width; x1 = width
    if y1 > height: y0 -= y1 - height; y1 = height
    return max(0, x0), max(0, y0), min(width, x1), min(height, y1)

@torch.no_grad()
def refine_proposal(image, proposal):
    """OOF実験 (V51 3-fold検証) と同一定義。shrink固定。"""
    x0, y0, x1, y1 = mask_square_box(proposal)
    crop = cv2.resize(image[y0:y1, x0:x1], (CFG.CROP_SIZE, CFG.CROP_SIZE), interpolation=cv2.INTER_LINEAR)
    tensor = torch.from_numpy(crop.transpose(2, 0, 1)[None].copy()).float().cuda() / 255.0
    probability = refiner(tensor).sigmoid()[0, 0].cpu().numpy()
    probability = cv2.resize(probability, (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR)
    refined = np.zeros_like(proposal, dtype=bool)
    refined[y0:y1, x0:x1] = probability >= REFINER_THRESHOLD
    return refined & proposal  # shrink固定(REFINER_METHOD="shrink")


In [ ]:
# [6/9] 排他化 (output/v35/run_pipeline_stack.py の apply_ownership と同一。変更なし)
def greedy_disjoint(masks, order, min_area=MIN_AREA):
    occupied = np.zeros((2048, 2048), dtype=bool)
    kept = []
    for index in order:
        exclusive = masks[index] & ~occupied
        if exclusive.sum() < min_area:
            continue
        kept.append(exclusive)
        occupied |= exclusive
    return kept

def drop_contained_indices(masks, threshold=0.90):
    if len(masks) < 2:
        return list(range(len(masks)))
    rles = mask_utils.encode(np.asfortranarray(np.stack(masks, axis=-1).astype(np.uint8)))
    areas = np.asarray(mask_utils.area(rles), dtype=np.float64)
    self_iou = np.asarray(mask_utils.iou(rles, rles, [0] * len(rles)), dtype=np.float64)
    dropped = set()
    for i in range(len(masks)):
        for j in range(i + 1, len(masks)):
            if i in dropped or j in dropped:
                continue
            value = self_iou[i, j]
            if value <= 0:
                continue
            intersection = value * (areas[i] + areas[j]) / (1.0 + value)
            smaller = i if areas[i] <= areas[j] else j
            if intersection / max(areas[smaller], 1.0) >= threshold:
                dropped.add(smaller)
    return [i for i in range(len(masks)) if i not in dropped]

def apply_ownership(masks, scores, rule=SUBMISSION_OWNERSHIP_RULE):
    if not masks:
        return []
    areas = np.array([m.sum() for m in masks], dtype=np.float64)
    if rule == "conf_first":
        return greedy_disjoint(masks, list(np.argsort(-scores)))
    if rule == "area_asc":
        return greedy_disjoint(masks, list(np.argsort(areas)))
    if rule == "drop_contained_then_area_asc":
        subset = drop_contained_indices(masks)
        return greedy_disjoint(masks, sorted(subset, key=lambda i: areas[i]))
    raise ValueError(rule)


In [ ]:
# [7/9] test 180枚に対して推論し、submission CSVを生成(単一モデル、アンサンブルなし)
rows, audit = [], []
for file_name in tqdm(test_files, desc="test推論(fold2単体 + EdgeAttNet refiner)"):
    image_path = CFG.PREPROC_DIR / file_name
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(f"前処理画像を読めない: {image_path}")
    proposals = single_fold_proposals(image_path)        # (confidence, mask) のリスト。アンサンブルなし
    scores = np.array([c for c, _ in proposals], dtype=np.float32)
    refined = [refine_proposal(image, m) for _, m in proposals]
    final_masks = apply_ownership(refined, scores)

    image_id = Path(file_name).stem
    for index, mask in enumerate(final_masks, start=1):
        encoded = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
        rows.append({"filament_id": f"{image_id}_{index}",
                     "segmentation_rle": encoded["counts"].decode("utf-8")})
    audit.append({"file_name": file_name, "proposals": len(proposals), "submitted": len(final_masks)})

submission = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
EXPERIMENT_ID = "v52_edgeattnet_refiner_fold2_only"
OUTPUT_PATH = Path(f"/content/submission_{EXPERIMENT_ID}.csv")
submission.to_csv(OUTPUT_PATH, index=False, lineterminator="\n")
DRIVE_OUT = CFG.ARTIFACT_ROOT / OUTPUT_PATH.name
shutil.copy2(OUTPUT_PATH, DRIVE_OUT)
pd.DataFrame(audit).to_csv(CFG.ARTIFACT_ROOT / f"{EXPERIMENT_ID}_audit.csv", index=False)
print({"rows": len(submission), "images_with_predictions": len(set(audit_row['file_name'] for audit_row in audit if audit_row['submitted'] > 0)),
       "drive_copy": str(DRIVE_OUT)})


In [ ]:
# [8/9] 公式RLE厳密検証 (提出前チェックリスト §4.4 に対応。V50と同一)
checked = pd.read_csv(OUTPUT_PATH, keep_default_na=False)
assert checked.columns.tolist() == ["filament_id", "segmentation_rle"]
assert len(checked) > 0
assert not checked.isna().any().any()
assert not checked["filament_id"].duplicated().any()
assert not checked["segmentation_rle"].str.strip().eq("").any()

test_ids = {Path(name).stem for name in test_files}
submitted_ids = {value.rsplit("_", 1)[0] for value in checked["filament_id"]}
assert submitted_ids <= test_ids

areas = []
occupancy = {}
for row in tqdm(checked.itertuples(index=False), total=len(checked), desc="decode + overlap検証"):
    decoded = mask_utils.decode({"size": [2048, 2048], "counts": row.segmentation_rle.encode("utf-8")})
    assert decoded.shape == (2048, 2048)
    area = int(decoded.sum())
    assert area > 0
    areas.append(area)
    image_id = row.filament_id.rsplit("_", 1)[0]
    occ = occupancy.setdefault(image_id, np.zeros((2048, 2048), dtype=bool))
    assert not np.logical_and(occ, decoded.astype(bool)).any(), f"重複: {image_id}"
    occ |= decoded.astype(bool)

print({"rows": len(checked), "images": len(submitted_ids), "area_min": min(areas),
       "area_median": int(np.median(areas)), "area_max": max(areas), "overlap_pixels": 0})


In [ ]:
# [9/9] manifest保存
import hashlib

def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

manifest = {
    "experiment_id": EXPERIMENT_ID,
    "purpose": "3-fold OOFで統計的に確認したEdgeAttNet refinerの改善(Δ+0.0028、"
               "95%CI[0.000915,0.004756]、frac_positive 99.8%)が、実際のPublic LBへ"
               "再現するかを確認",
    "diff_vs_v50": "refinerをMediSegUNetからMediSegEdgeAttUNet(EdgeGuidedMHSA"
                   "ボトルネック付き、+4.2Mパラメータ)へ差し替え。YOLOモデル"
                   "(fold2単体)・後処理はV50と完全同一",
    "oof_validation": "v51_edgeattnet_refiner_3fold_manifest.json 参照。"
                       "fold0/1/2全て同符号(正)、3-fold全結合(n=707)で有意。",
    "single_fold": CFG.SINGLE_FOLD,
    "metric_definition_version": "official-v1-2026-08-15",
    "submission_conf": SUBMISSION_CONF,
    "ownership_rule": SUBMISSION_OWNERSHIP_RULE,
    "refiner_method": REFINER_METHOD,
    "refiner_threshold": REFINER_THRESHOLD,
    "refiner_architecture": "MediSegEdgeAttUNet (all_data, warm-started from fold0)",
    "min_area": MIN_AREA,
    "compare_against": {"experiment_id": "v50_no_ensemble_fold2_only", "kaggle_public_score": 0.37},
    "rows": len(checked),
    "submission_sha256": sha256_of(OUTPUT_PATH),
    "kaggle_public_score": "unknown",
    "decision": "pending",
}
(CFG.ARTIFACT_ROOT / f"{EXPERIMENT_ID}_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("\n次: ローカルへ submission_v52_edgeattnet_refiner_fold2_only.csv をダウンロードし、"
      "kaggle competitions submit で提出。")
